In [1]:
import torch
from torch import nn

### Exercise 1

What happens if you specify the input dimensions to the first layer but not to subsequent layers? Do you get immediate initialization?

In [2]:
net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.LazyLinear(10)
)

print("First layer weight:")
print(net[0].weight.shape)

print("\nThird layer weight before forward pass:")
print(net[2].weight)

First layer weight:
torch.Size([256, 20])

Third layer weight before forward pass:
<UninitializedParameter>


/Users/zouminghao/Desktop/d2l-notes-exercises/venv/lib/python3.11/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


If the input dimension of the first layer is specified explicitly, that layer is initialized immediately because its parameter shape is already known. However, subsequent lazy layers are not initialized immediately unless their input dimensions are also fully specified. They remain uninitialized until the first forward pass, when the framework infers their input shapes from the actual data flowing through the network.

### Exercise 2

What happens if you specify mismatching dimensions?

In [4]:
X = torch.rand(2, 20)

layer1 = nn.Linear(20, 256)
relu = nn.ReLU()
layer2 = nn.Linear(128, 10)   # intentionally wrong

h1 = layer1(X)
print("After layer1:", h1.shape)

h2 = relu(h1)
print("After ReLU:", h2.shape)

out = layer2(h2)   # error happens here
print("After layer2:", out.shape)

After layer1: torch.Size([2, 256])
After ReLU: torch.Size([2, 256])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x256 and 128x10)

If I specify mismatching dimensions, the model may still be constructed successfully, because each layer is valid on its own. However, during the forward pass, PyTorch raises a runtime error when one layer’s output dimension does not match the next layer’s expected input dimension.

### Exercise 3

What would you need to do if you have input of varying dimensionality? Hint: look at the parameter tying.

If the input dimensionality varies, a standard linear layer cannot directly handle all inputs, because once its parameters are initialized, the weight matrix has a fixed shape. Lazy initialization only postpones choosing that shape until the first forward pass; it does not allow the shape to keep changing afterward. Therefore, to handle varying dimensionality, we would need to convert the inputs into a common fixed-size representation before applying shared parameters. This can be done, for example, by padding inputs to the same size, or by applying a shared transformation to fixed-size local parts of the input and then pooling them into a fixed-dimensional summary. If we do not standardize the representation, then we would need separate parameter sets for different input sizes, and full parameter tying would no longer be possible.